# M4 Agentic AI - Component-level evals for the AIAgentPractice projects

## 1. Introduction
In the M4 lab you added a **component-level eval** to a research workflow: instead of judging the final essay, you checked one component — the web-search step — with an objective, per-example check (are the URLs from preferred domains?), returning a PASS/FAIL flag and a Markdown summary.

This notebook applies the same idea to every project built so far:

| Project | Component evaluated | Ground truth per example | Metric |
|---|---|---|---|
| SqlQueryImprover | `generate_sql` (V1) and `refine_sql_external_feedback` (V2) | hand-written correct SQL | execution accuracy |
| ChartGenerationAgent | `generate_chart_code` (V1) and `reflect_on_image_and_regenerate` (V2) | expected numbers computed from the CSV | charts passing every check |
| FunctionsToTools | the LLM's tool selection | expected tools, order and arguments | examples fully correct |
| EmailTool | the email agent (LLM + 10 tools) | expected change to the inbox | requests handled correctly |

All four are **objective evals with a per-example ground truth** (the upper-left quadrant from the lecture): no LLM-as-judge, just code. Each eval has an `evaluate_*` function for one example (like `evaluate_tavily_results`) and a `run_eval()` that scores the whole set against a `min_ratio` threshold.

### Why component-level?
- Re-running a whole pipeline to test one step is expensive and noisy; later steps add randomness.
- A per-component metric gives a clear signal: did *this* change (prompt, model, tool docstring) help?
- For the reflection projects we score **V1 and V2 on the same examples**, which tells us whether the reflection step is worth its cost.

> **Before you start:** the evals call real models. Put `OPENAI_API_KEY` in `Eval/.env` (see `.env.example`) — or the `.env` you already created for ReflectDesinPatternDemo / ToolUseDemo is picked up automatically. Use `limit=2` for a quick, cheap run.

## 2. Setup

In [ ]:
import pandas as pd

import eval_utils as eu
import sql_eval
import chart_eval
import tool_selection_eval
import email_agent_eval

LIMIT = None   # e.g. 2 → only the first 2 examples of each eval (quick and cheap)
results = {}

## 3. SqlQueryImprover — execution accuracy
Each question has a hand-written correct query. The agent's SQL is executed and compared with the ground-truth result. The comparison ignores column names/order, row order, extra columns and float rounding — only the answer matters.

Several questions contain the traps from the M2 lab: sales stored as **negative** `qty_delta`, a product whose latest event is a restock with a **NULL price**, and stock that must include the **initial insert**.

In [ ]:
pd.DataFrame(sql_eval.EVAL_SET)[["id", "question", "tests"]]

The per-example check (compare a result with the ground truth):

In [ ]:
import inspect
print(inspect.getsource(sql_eval.compare_results))

In [ ]:
results["sql"] = sql_eval.run_eval(
    model_generation="openai:gpt-4.1",
    model_evaluation="openai:gpt-4.1",
    min_ratio=0.8,
    limit=LIMIT,
)

## 4. ChartGenerationAgent — chart correctness
For each request the expected numbers are computed from `coffee_sales.csv` with pandas. The generated code is executed, the figure is captured when it is saved, and the numbers it actually draws (bar heights, line points, pie slices, value labels) are compared with the expected ones. Ambiguous requests (*"Q1 coffee sales"*: revenue? number of sales? per coffee?) list every reasonable reading.

Checks per chart: `code_extracted` · `runs` · `saved_to_path` · `title` · `axis_labels` · `rules` (no `plt.show`, no seaborn) · `correct_data` (≥ 90% of expected values found).

In [ ]:
_, chart_utils = chart_eval.load_chart_project()
coffee = chart_utils.load_and_prepare_data(str(eu.PROJECTS["chart"] / "coffee_sales.csv"))

pd.DataFrame([{"id": e["id"], "instruction": e["instruction"],
               "accepted readings": ", ".join(e["expected"](coffee))} for e in chart_eval.EVAL_SET])

In [ ]:
results["chart"] = chart_eval.run_eval(
    generation_model="gpt-4o-mini",
    reflection_model="o4-mini",
    min_ratio=0.6,
    limit=LIMIT,
)

The V1 and V2 charts are saved in `results/charts/` — open a pair to see what the reflection step changed.

## 5. FunctionsToTools — tool selection & arguments
The project's four tools are replaced by **recording fakes** with the same name, signature and docstring, so the LLM sees exactly the same tools — but nothing is written and no network calls are made. The fakes return fixed values (time `14:05:09`, weather `72.5°F`) so we can check that a result flows into the next call (e.g. the weather ends up in the note).

The set includes a prompt that needs **no tool** — calling one anyway is a failure.

In [ ]:
pd.DataFrame([{"id": e["id"], "prompt": e["prompt"], "expected tools": ", ".join(e["tools"]) or "—",
               "order": e.get("order", ""), "argument rules": e.get("args", "")} for e in tool_selection_eval.EVAL_SET])

In [ ]:
results["tools"] = tool_selection_eval.run_eval(model="openai:o4-mini", min_ratio=0.8, limit=LIMIT)

## 6. EmailTool — correct inbox changes
For each request the sample inbox is reset, the agent runs with all ten tools, and the inbox before/after is compared. The agent must make the expected change **and nothing else** — no deleting the wrong email, no stray replies, no marking unrelated emails as read.

This runs on its own copy of the inbox (`results/emails_eval.db`, port 8766), so your EmailTool inbox is never touched.

In [ ]:
pd.DataFrame([{"id": e["id"], "request": e["request"],
               "deleted": sorted(e.get("deleted", [])), "marked read": sorted(e.get("marked_read", [])),
               "sent to": [r for r, _ in e.get("sent", [])], "answer": e.get("answer", "")}
              for e in email_agent_eval.EVAL_SET])

In [ ]:
results["email"] = email_agent_eval.run_eval(model="openai:gpt-4.1", min_ratio=0.75, limit=LIMIT)

## 7. Summary

In [ ]:
summary = pd.DataFrame([
    {"component": r.title, "metric": r.metric, "score": f"{r.score:.0%}",
     "threshold": f"{r.threshold:.0%}", "status": eu.status(r.passed)}
    for r in results.values()
])
summary

Save the reports (Markdown + JSON per eval) to `results/` so you can compare runs over time:

In [ ]:
for r in results.values():
    print(eu.save_result(r))

## 8. Try it yourself
- **Compare models:** re-run one eval with another model (e.g. `tool_selection_eval.run_eval(model="openai:gpt-4.1-mini")`) and compare the scores.
- **Is reflection worth it?** In the SQL and chart reports, look at *V1 → V2*: how many examples did reflection fix, and how many did it break?
- **Grow the eval set:** add an example to an `EVAL_SET` — e.g. a new SQL question with its ground-truth query, or an email request with its expected inbox change. ~10–20 examples per component give a much more reliable signal.
- **Change the bar:** adjust `min_ratio` to what "good enough" means for that component.
- **Improve the component, re-run the eval:** e.g. tweak a tool docstring in `email_tools.py` or the prompt in `generate_sql`, then check whether the score moves.

## 9. Takeaways
- Each project now has a **component-level, objective eval** with a per-example ground truth — cheap to run, reproducible, and independent of the rest of the pipeline.
- Scoring **V1 and V2** separately turns "reflection seems to help" into a number.
- Evals on **side effects** (the inbox) catch agents that do the task *and* something they shouldn't.
- Faking tools (FunctionsToTools) isolates one component — the LLM's tool choice — from slow or side-effecting tools.
- Track the scores over time (`results/`) as you change prompts, models or tools.